data gen

In [ ]:
import requests
import pandas as pd
import numpy as np
import os

os.makedirs("data", exist_ok=True)

url = "https://power.larc.nasa.gov/api/temporal/daily/point"

params = {
    "parameters": "WS2M,T2M",
    "community": "RE",
    "latitude": 11.92237,
    "longitude": 79.60673,
    "start": 20250101,
    "end": 20251231,
    "format": "JSON"
}

response = requests.get(url, params=params)
data = response.json()

wind = data["properties"]["parameter"]["WS2M"]
temp = data["properties"]["parameter"]["T2M"]

rows = []
prev_eff = 0

max_wind = max(wind.values())  # for scaling

for date in wind.keys():

    wind_speed = wind[date]
    temperature = temp[date]

    # simulate hourly variation
    for hour in range(24):

        # wind fluctuation
        fluctuation = np.random.uniform(0.7, 1.3)
        wind_hour = wind_speed * fluctuation

        # ⚡ wind power ~ cube relation
        power = wind_hour ** 3

        efficiency = (power / (max_wind ** 3)) * 100
        efficiency = max(0, min(efficiency, 100))

        rows.append([
            date,
            hour,
            temperature,
            wind_hour,
            prev_eff,
            round(efficiency, 2)
        ])

        prev_eff = efficiency

df = pd.DataFrame(rows, columns=[
    "date",
    "hour",
    "temperature",
    "wind_speed",
    "historical_efficiency",
    "efficiency"
])

df.to_csv("data/wind.csv", index=False)

print("🌬️ Wind dataset created!")

🌬️ Wind dataset created!


In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np
import joblib

df = pd.read_csv("data/wind.csv")

# extract month
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month

features = [
    "month",
    "hour",
    "temperature",
    "wind_speed",
    "historical_efficiency"
]

X = df[features]
y = df["efficiency"]

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=200, max_depth=12)
model.fit(X_train, y_train)

# evaluate
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
acc = (1 - mae / 100) * 100

print("\n🌬️ WIND MODEL PERFORMANCE")
print("MAE:", round(mae, 2))
print("R2 :", round(r2, 3))
print("acc:", acc)

joblib.dump(model, "models/wind.pkl")

print("\n✅ Wind model trained!")


🌬️ WIND MODEL PERFORMANCE
MAE: 0.01
R2 : 1.0
acc: 99.99223036299533

✅ Wind model trained!


In [1]:
import joblib
import pandas as pd

model = joblib.load("models/wind.pkl")

PANEL_CAPACITY_KW = 2.0  # wind turbine capacity

input_data = {
    "month": 5,
    "hour": 14,
    "temperature": 30,
    "wind_speed": 10,
    "historical_efficiency": 0
}

df = pd.DataFrame([input_data])

efficiency = model.predict(df)[0]
efficiency = max(0, min(efficiency, 100))

power_output = (efficiency / 100) * PANEL_CAPACITY_KW

print(f"\n🌬️ Efficiency: {round(efficiency,2)} %")
print(f"⚡ Wind Power: {round(power_output,2)} kW")


🌬️ Efficiency: 99.09 %
⚡ Wind Power: 1.98 kW


In [16]:
import joblib
import pandas as pd

model = joblib.load("models/wind.pkl")

PANEL_CAPACITY_KW = 2.0

scenarios = [
    ("🌬️ Low Wind", 2),
    ("🌬️ Medium Wind", 5),
    ("🌪️ High Wind", 10)
]

for name, wind_speed in scenarios:

    df = pd.DataFrame([{
        "month": 5,
        "hour": 12,
        "temperature": 30,
        "wind_speed": wind_speed,
        "historical_efficiency": 40
    }])

    eff = model.predict(df)[0]
    eff = max(0, min(eff, 100))

    power = (eff / 100) * PANEL_CAPACITY_KW

    print(f"\n{name}")
    print(f"Efficiency: {round(eff,2)} %")
    print(f"Power     : {round(power,2)} kW")


🌬️ Low Wind
Efficiency: 3.56 %
Power     : 0.07 kW

🌬️ Medium Wind
Efficiency: 55.59 %
Power     : 1.11 kW

🌪️ High Wind
Efficiency: 99.32 %
Power     : 1.99 kW
